# PR 六个评分分布

相同候选池；全文/答案两个监督目标。读取已完成的真实评分，不运行模型、不选数、不读取测试 loss。

In [ ]:
%matplotlib inline
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
root=Path.cwd()
scores=pd.read_parquet(root/'scores.parquet')
import json
configuration=json.loads((root/'effective_config.json').read_text())
methods=configuration['methods']
objectives=['full_text','answer_only']
figures=root/'figures'
figures.mkdir(exist_ok=True)
display(pd.read_csv(root/'distribution_statistics.csv'))

In [ ]:
fig,axes=plt.subplots(2,len(methods),figsize=(5*len(methods),8),squeeze=False)
for i,obj in enumerate(objectives):
    group=scores[scores.objective==obj]
    for j,name in enumerate(methods):
        ax=axes[i,j]
        ax.hist(group[name],bins=100,log=True)
        ax.axvline(0,color='black',linestyle='--',linewidth=.8)
        ax.set(title=f'{obj} / {name}',xlabel='Raw score',ylabel='Count (log scale)')
fig.tight_layout()
fig.savefig(figures/'six_distributions.png',dpi=180)
plt.show()

In [ ]:
fig,axes=plt.subplots(2,len(methods),figsize=(5*len(methods),8),squeeze=False)
for i,obj in enumerate(objectives):
    group=scores[scores.objective==obj]
    for j,name in enumerate(methods):
        ax=axes[i,j]
        x=np.sort(group[name].to_numpy())
        ax.plot(x,np.arange(1,len(x)+1)/len(x))
        ax.axvline(0,color='black',linestyle='--',linewidth=.8)
        ax.set(title=f'{obj} / {name}',xlabel='Raw score',ylabel='Empirical CDF')
fig.tight_layout()
fig.savefig(figures/'six_ecdfs.png',dpi=180)
plt.show()

In [ ]:
fig,axes=plt.subplots(2,len(methods),figsize=(5*len(methods),8),squeeze=False)
for i,obj in enumerate(objectives):
    group=scores[scores.objective==obj]
    for j,name in enumerate(methods):
        ax=axes[i,j]
        ax.scatter(group.input_tokens,group[name],s=3,alpha=.25)
        ax.set(title=f'{obj} / {name}',xlabel='Input tokens',ylabel='Score')
fig.tight_layout()
fig.savefig(figures/'length_vs_scores.png',dpi=180)
plt.show()

In [ ]:
fig,axes=plt.subplots(1,len(methods),figsize=(5*len(methods),4),squeeze=False)
a=scores[scores.objective=='full_text'].set_index('id')
b=scores[scores.objective=='answer_only'].set_index('id').loc[a.index]
for ax,name in zip(axes[0],methods):
    ax.scatter(a[name].rank(pct=True),b[name].rank(pct=True),s=3,alpha=.2)
    ax.set(title=name,xlabel='Full-text percentile rank',ylabel='Answer-only percentile rank')
fig.tight_layout()
fig.savefig(figures/'objective_rank_comparison.png',dpi=180)
plt.show()
for obj in objectives:
    display(pd.read_csv(root/f'correlations_{obj}.csv',index_col=0))

In [ ]:
fig,axes=plt.subplots(2,len(methods),figsize=(5*len(methods),8),squeeze=False)
for i,obj in enumerate(objectives):
    group=scores[scores.objective==obj]
    for j,name in enumerate(methods):
        ordered=group.sort_values([name,'id'],ascending=[False,True])
        x=np.arange(1,len(ordered)+1)/len(ordered)
        ax=axes[i,j]
        ax.plot(x,ordered.input_tokens.cumsum(),label='Input tokens')
        ax.plot(x,ordered.prediction_tokens.cumsum(),label='Supervised tokens')
        ax.set(title=f'{obj} / {name}',xlabel='Fraction of records',ylabel='Cumulative tokens')
        ax.legend()
fig.tight_layout()
fig.savefig(figures/'ranked_token_budgets.png',dpi=180)
plt.show()